# Modified VIEScore for RefEdit-Bench (Text Prompts)

Scores **existing generated RefEdit outputs**; this notebook does not run the editing agent. This notebook scores the `text` inference run family. Set `MMEDIT_REFEDIT_BENCH_TEXT_RUN` to select a specific run, or leave it empty to score the latest run. It reads that run's `results/results.jsonl` and `results/images/*_output.png`, so saved images remain discoverable if a subset run replaced the JSONL summary. Scoring uses RefEdit's `metadata.edit_instruction_single`, the dataset GT polygon mask, and identical zero-padding tight source/output crops. Scores are stored separately in that run's `metrics/modified_viescore.jsonl`; inference never creates this directory. 

Install GEdit-Bench VIEScore dependencies in this notebook's Python environment (including `regex` and `requests`), clone [Step1X-Edit](https://github.com/stepfun-ai/Step1X-Edit), and set `VIESCORE_ROOT` to its `GEdit-Bench` directory. Set `VIESCORE_KEY_PATH` to a local file containing only your OpenAI API key on the first line. The scorer requests the `gpt-4o` API model; API usage may incur charges. Never put the key in a cell or commit its file.

In [1]:
import os
from pathlib import Path

from evaluation.notebook import load_benchmark, resolve_run_directory

BENCHMARK = "refedit_bench"
PROMPT_MODE = "text"
context = load_benchmark(BENCHMARK, prompt_mode=PROMPT_MODE)
ANNOTATION_FILE = context.annotation_file
OUTPUT_ROOT = context.output_dir
RUN_NAME = os.environ.get("MMEDIT_REFEDIT_BENCH_TEXT_RUN") or None
OUTPUT_DIR = resolve_run_directory(OUTPUT_ROOT, RUN_NAME)
records = context.records
image_loader = context.image_loader
if not OUTPUT_DIR.is_dir():
    raise FileNotFoundError(f"RefEdit output directory not found: {OUTPUT_DIR}")
VIESCORE_ROOT = Path(os.environ.get("VIESCORE_ROOT") or ".").expanduser()
VIESCORE_KEY_PATH = Path(os.environ.get("VIESCORE_KEY_PATH") or ".").expanduser()
print("Annotations:", ANNOTATION_FILE)
print("Selected run:", OUTPUT_DIR)
print("VIEScore checkout:", VIESCORE_ROOT if str(VIESCORE_ROOT) != "." else "not configured")
print("API key file:", VIESCORE_KEY_PATH if str(VIESCORE_KEY_PATH) != "." else "not configured")

Annotations: /Users/nghialt3670/Projects/MMEdit/mmedit-framework/.out/annotation/refedit_bench/refedit_bench_mixed.jsonl
Selected run: /Users/nghialt3670/Projects/MMEdit/mmedit-framework/.out/evaluation/refedit_bench/text/run_20261011_163954_808537
VIEScore checkout: /Users/nghialt3670/Projects/Step1X-Edit/GEdit-Bench
API key file: /Users/nghialt3670/Projects/MMEdit/mmedit-framework/.viescore.key


In [2]:
from evaluation.common import HFImageLoader, read_jsonl
from evaluation.refedit_bench_viescore import collect_saved_outputs

records = list(read_jsonl(ANNOTATION_FILE))
if not records:
    raise ValueError(f'No RefEdit annotations found in {ANNOTATION_FILE}')
image_loader = HFImageLoader()
saved_outputs = collect_saved_outputs(
    records,
    OUTPUT_DIR / 'results' / 'results.jsonl',
    OUTPUT_DIR / 'results' / 'images',
    prompt_mode=PROMPT_MODE,
)
print(f'Found {len(saved_outputs)} saved output images:')
for saved in saved_outputs:
    print(f"  [{saved['annotation_index']}] {saved['sample_id']} ({saved['discovered_from']})")

Found 200 saved output images:
  [0] easy:000 (results.jsonl)
  [1] easy:001 (results.jsonl)
  [2] easy:002 (results.jsonl)
  [3] easy:003 (results.jsonl)
  [4] easy:004 (results.jsonl)
  [5] easy:005 (results.jsonl)
  [6] easy:006 (results.jsonl)
  [7] easy:007 (results.jsonl)
  [8] easy:008 (results.jsonl)
  [9] easy:009 (results.jsonl)
  [10] easy:010 (results.jsonl)
  [11] easy:011 (results.jsonl)
  [12] easy:012 (results.jsonl)
  [13] easy:013 (results.jsonl)
  [14] easy:014 (results.jsonl)
  [15] easy:015 (results.jsonl)
  [16] easy:016 (results.jsonl)
  [17] easy:017 (results.jsonl)
  [18] easy:018 (results.jsonl)
  [19] easy:019 (results.jsonl)
  [20] easy:020 (results.jsonl)
  [21] easy:021 (results.jsonl)
  [22] easy:022 (results.jsonl)
  [23] easy:023 (results.jsonl)
  [24] easy:024 (results.jsonl)
  [25] easy:025 (results.jsonl)
  [26] easy:026 (results.jsonl)
  [27] easy:027 (results.jsonl)
  [28] easy:028 (results.jsonl)
  [29] easy:029 (results.jsonl)
  [30] easy:030 (re

## Score saved outputs

Run this cell after reviewing the discovered sample list above. Unchanged images reuse cached VIEScore rows; changed or newly discovered images are evaluated and appended to the output JSONL.

## Configure VIEScore

This uses the upstream VIEScore `tie` task with the GPT-4o API model. The API key is read from the local key file; it is never displayed.

## Score saved outputs

The loop below checks annotation/dataset consistency, reuses unchanged cached rows, scores uncached outputs, and atomically rewrites the JSONL after each sample so progress is preserved.

In [3]:
from IPython.display import display
from evaluation.refedit_bench_viescore import evaluate_saved_refedit_bench_outputs

metric_rows, score_summary = evaluate_saved_refedit_bench_outputs(
    records,
    output_dir=OUTPUT_DIR,
    image_loader=image_loader,
    viescore_root=VIESCORE_ROOT,
    key_path=VIESCORE_KEY_PATH,
    prompt_mode=PROMPT_MODE,
)
print(f"Per-sample Modified VIEScore: {OUTPUT_DIR / 'metrics' / 'modified_viescore.jsonl'}")
display([{"category": category, **metrics} for category, metrics in score_summary.items()])

Modified VIEScore:   0%|          | 0/200 [00:00<?, ?sample/s]

Scored 200 saved RefEdit outputs (100 easy, 100 hard).
Per-sample Modified VIEScore: /Users/nghialt3670/Projects/MMEdit/mmedit-framework/.out/evaluation/refedit_bench/text/run_20261011_163954_808537/metrics/modified_viescore.jsonl
Per-sample Modified VIEScore: /Users/nghialt3670/Projects/MMEdit/mmedit-framework/.out/evaluation/refedit_bench/text/run_20261011_163954_808537/metrics/modified_viescore.jsonl


[{'category': 'easy',
  'sc': 7.85,
  'pq': 4.36,
  'o': 5.362611413846014,
  'count': 100},
 {'category': 'hard',
  'sc': 6.92,
  'pq': 4.39,
  'o': 4.733555890027455,
  'count': 100}]